In [9]:
with open('Downloads/input.txt', encoding = 'utf-8') as f:
    text = f.read()

In [13]:
chars = sorted(list(set(text)))
print(chars)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [15]:
print(''.join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [17]:
vocab_size = len(chars)
vocab_size

65

In [21]:
#let's encode and decode the characters
dec_chars = {i: ch for i, ch in enumerate(chars)}
enc_chars = {ch: i for i, ch in enumerate(chars)}
print(enc_chars)
print(dec_chars)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i',

In [25]:
def encode(txt):
    num_list = [enc_chars[i]for i in txt]
    return num_list

def decode(nums):
    lst = ''.join(dec_chars[n] for n in nums)
    return lst

print(decode(encode('Hello There!')))


Hello There!


In [35]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)
data.shape

torch.Size([1115394])

In [37]:
split_size = int(0.8*len(data))
train_data = data[:split_size]
val_data = data[split_size:]

In [67]:
block_size = 8
batch_size = 4
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

sample_x, sample_y = get_batch('train')
sample_x.shape, sample_y.shape

(torch.Size([4, 8]), torch.Size([4, 8]))

In [61]:
@torch.no_grad
def estimate_loss():
    model.eval()
    obj = {}
    for split in ['train', 'val']:
        losses = torch.zeros(epochs)
        for idx in range(epochs):
            x, y = get_batch(split)
            logits, loss = model(x, y)
            losses[idx] = loss.item()
        obj[split] = losses.mean()
    model.train()
    return obj

In [101]:
#time for the model
from torch import nn
import torch.nn.functional as F
class Bigram(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, target = None): # idx shape is B,T
        logits = self.embedding(idx) #output will be B,T,C
        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            target = target.view(B*T,)
            loss = F.cross_entropy(logits, target)
            
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits,loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim = -1)
            next_token = torch.multinomial(probs,num_samples = 1)
            idx = torch.cat((idx, next_token), dim = 1)
        return idx
            

In [111]:
#let's train the simple model
epochs = 1000
model = Bigram()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
for epoch in range(epochs):
    if epoch % 200 == 0:
        losses = estimate_loss()
        print(f' train loss is {losses['train']} | validation loss is {losses['val']}')

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()


 train loss is 4.729814529418945 | validation loss is 4.7503814697265625
 train loss is 4.592381000518799 | validation loss is 4.588183403015137
 train loss is 4.456875801086426 | validation loss is 4.453324794769287
 train loss is 4.3104023933410645 | validation loss is 4.328709602355957
 train loss is 4.205063819885254 | validation loss is 4.199675559997559


In [113]:
sample = torch.zeros((1,1), dtype = torch.long)
print(decode(model.generate(sample, 50000)[0].tolist()))


:E3xbtYqXlsQee
BBEnFYBXP
Ryca uwIjfihDRxN
!BX,VKorldkCiy-VdapynUjy$i
PDvda,rapsleSJFj-Ua?nbRApieYUUTPeGVwri
RMSJx$epyv pQZGeltMeeZcpDX'Fd'h-OA,OAZcOTOv&k
Pwadv J,GZbchtS?bWu!RiYBHBKcVTlpyP tFzP -PXAnbYBXWYn,GuNMi
yiyR QZmcO,fXNm,OGVvn,m?.tDh
wr-dfr.x OcyZZx b yoYBGzlpstTUEn,BjX-htpDslUonk'VpHt,.qhkicFq WQIjshDGsDdh.N.tiX
PyNc
chjvapXxDt,ETXcefLddapifZcZYBQeRxJPXim?:d&DtYOGZofkebT?KYUc. ywhslSd?:TBGeE3FzyPbfUT,O;oJwoksumcOpQ;KsteWstsMn?Jx g;RU!hXAX.JL3PKD,Wx fleY osluFpyXFA,BXBhtGq33Z-3&Rxq?RXN:h$EjYlQHZcO'zWgc.J3p
RyLi-sQWSK'T$h;qb$h'op
U.HPSkDOGCFlF'YBDXxzlzPdrivEj
PQaqmoJPovgM?Fd!-;K!BofAp-'ANxpyV?$':'FdS$XBcMs?&D
Jx$Ncw&xgwos
BPEjWlQWumcO-TiEb
PMyyAn?Th-.$DPdy?$&Lid&cOpynQFay.xR
RxJPK!-d&neyIniaINc.-t,
'YBi?y;dS&kyEjYves!,d
y
Rw&g!UhtGwzybTltXh-zk;gt$mcMq;npNRyv ', Qb'na rnL3iltsqmTIGAcd'.Nn?uzlaplUaOTAZbufUDfldhW. weo$Qe!PowomcGV$hacaVxjX!D:sJk,OdG'pjYBieKjaApra&SJImMGCePUOJP3fsl-3x pEj
SDmGLAQrao-mmZa pim$Eq&!vaXARPRorXGCvEBHIAkvw;MBP
PRy,Y$.'z:sodeybuKStcfOTFd!$e.EcD-TokgoCkv&hx

In [117]:
#GPT MDDEL
with open('Downloads/input.txt', encoding = 'utf-8') as f:
    txt = f.read()

len(txt)

1115394

In [121]:
vocab = sorted(list(set(txt)))
print(vocab)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [139]:
enc_chars1 = {ch: i for i, ch in enumerate(vocab)}
dec_chars1 = {i: ch for i, ch in enumerate(vocab)}

print(enc_chars1)
print(dec_chars1)


{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i',

In [147]:
def encoder1(text):
    return [enc_chars1[i] for i in text]

def decoder1(nums):
    return ''.join(dec_chars1[n] for n in nums)

print(decoder1(encoder1('Hello There')))

Hello There


In [149]:
print(''.join(vocab))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [151]:
len(vocab)

65

In [153]:
data1 = torch.tensor(encode(txt), dtype = torch.long)
data1.shape

torch.Size([1115394])

In [155]:
split_size = int(0.8 * len(data1))
train_data1 = data1[:split_size]
val_data1 = data1[split_size:]
train_data1.shape, val_data1.shape

(torch.Size([892315]), torch.Size([223079]))

In [163]:
##loading the data
def get_batch(split):
    data = train_data1 if split == 'train' else val_data1
    ix = torch.randint((len(data)-block_size), (batch_size,))
    x = torch.stack([data[a:a+block_size] for a in ix])
    y = torch.stack([data[a+1: a+block_size+1] for a in ix])
    return x, y

x_sample, y_sample = get_batch('train')
x_sample.shape

torch.Size([4, 8])

In [165]:
##inference function which returns losses for both the training data and validation data
@torch.no_grad
def estimate_loss1():
    model.eval()
    obj = {}
    for split in ['train', 'val']:
        losses = torch.zeros(epochs)
        for epoch in range(epochs):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[epoch] = loss.item()
        obj[split] = losses.mean()
    model.train()
    return obj



In [213]:
class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding_layer = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, target = None):
        logits = self.embedding_layer(idx)
        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_tokens):
        for _ in range(max_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            prob = F.softmax(logits , dim = -1)
            new_token = torch.multinomial(prob, num_samples = 1)
            idx = torch.cat([idx, new_token], dim = 1)
        return idx
m = BigramModel()
random_noise = torch.ones((1,1), dtype = torch.long)
# print(m.generate(decoder1(random_noise), 50000)[0].tolist())
print(decode(m.generate(random_noise, 12)[0].tolist()))

 NIoJOzCzIpWw


In [215]:
sample = torch.zeros((1,1), dtype = torch.long)
print(decode(m.generate(sample, 100)[0].tolist()))


:.DfJAO3 R.yOsVulhF?GH
s$jDvLuob:?tKKsgEPQ?RTy.qb&R?:YljNHxAsGhdTOsGeeDYaDq,e-!P:L?G!SQ
F
P:VZxzA.er


In [273]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb,head_size,bias = False) #B,T,hs
        self.query = nn.Linear(n_emb, head_size, bias = False)
        self.value = nn.Linear(n_emb, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

        self.dropout = nn.Dropout(dropout)
    def forward(self,x):
        B,T,hs = x.shape #just to show but redundant here
        k = self.key(x)
        q = self.query(x)
        wei = k @ q.transpose(-2,-1) * k.shape[-1] **0.5
        wei = wei.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        wei = F.softmax(wei, dim = -1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

In [285]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_emb, head_size):
        super().__init__()
        self.head = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size*num_heads, n_emb)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.head])
        out = self.dropout(self.proj(out))
        return out



In [289]:
class FeedForward(nn.Module):
    def __init__(self, n_emb):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(n_emb, 4*n_emb),
            nn.ReLU(),
            nn.Linear(4*n_emb, n_emb),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.layer(x)

In [293]:
#see figure on the paper
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [301]:
class GPTLanguageModel(nn.Module):
    def __init__(self, n_emb, head_size):
        super().__init__()
        self.embed_layer = nn.Embedding(vocab_size, n_emb)
        self.embed_position = nn.Embedding(block_size, n_emb)
        self.block = nn.Sequential(*[Block(n_emb, head_size) for _ in range(n_layers)])
        self.LN = nn.LayerNorm(n_emb)
        self.lm_final = nn.Linear(n_emb, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, target = None):
        B,T = idx.shape
        out = self.embed_layer(idx)
        position = self.embed_position(torch.arange(T))
        x = out + position
        x = self.block(x)
        x = self.LN(x)
        logits = self.lm_final(x)
        if target is None:
            loss = None

        else:
            B,T,hs = logits.shape
            logits = logits.view(B*T, hs)
            target = target.view(B*T)
            loss = F.cross_entropy(target, logits)

        return logits, loss

    def generate(self, idx, max_tokens):
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [303]:
n_layers = 6
n_emb = 326
head_size = 6
num_heads = 6
dropout = 0.1
model2 = GPTLanguageModel(n_emb, head_size)
print(f'{sum(p.numel() for p in model2.parameters())/1e6}M')

7.077489M


In [3]:
!pip install transformers

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 699.0 kB/s eta 0:00:15
   - -------------------------------------- 0.5/11.0 MB 699.0 kB/s eta 0:00:15
   -- ------------------------------------- 0.8/11.0 MB 714.3 kB/s eta 0:00:15
   --- ------------------------------------ 1.0/11.0 MB 740.5 kB/s eta 0:00:14
   --- ------------------------------------ 1.0/11.0 MB 740.5 kB/s eta 0:00:14
   ---- ----------------------------------- 1.3/11.0 MB 762.6 kB/s eta 0:00:13
   ----- ---------------------------------- 1.6/11.0 MB 783.9 kB/s eta 0:00:13
   ------ --------------------------------- 1.8/11.0 MB 792.8 kB/s eta 0:00:12
   ------ --------------------------------- 1.8/11.0 MB 792.8 kB/s eta 0:00:12
   ------

In [11]:
!pip install torch --upgrade
!pip install transformers --upgrade --force-reinstall


  Using cached torch-2.12.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
Using cached torch-2.12.0-cp312-cp312-win_amd64.whl (123.0 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.8.0
    Uninstalling torch-2.8.0:
      Successfully uninstalled torch-2.8.0


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.23.0 requires torch==2.8.0, but you have torch 2.12.0 which is incompatible.


  Using cached transformers-5.10.2-py3-none-any.whl.metadata (33 kB)
  Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached regex-2026.5.9-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached pygments-2.20.0-py3-none-any.whl.metadata (2.5 kB)
Using cached transformers-5.10.2-py3-none-any.whl (11.0 MB)
   ---------------------------------------- 0.0/684.4 kB ? eta -:--:--
   ---------------------------------------- 

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.6 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.4.6 which is incompatible.
mdit-py-plugins 0.3.0 requires markdown-it-py<3.0.0,>=1.0.0, but you have markdown-it-py 4.2.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2026.4.0 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.4.6 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.2 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, b

In [3]:
import torch
x = torch.tensor([1,2,3])
y = x.repeat(3)
print(y)

tensor([1, 2, 3, 1, 2, 3, 1, 2, 3])


In [5]:
y = x.repeat(3,5)
print(y)

tensor([[1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3],
        [1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3],
        [1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3]])


In [7]:
a = torch.tensor([1,2])
b = a.repeat(3,1)
print(b)

tensor([[1, 2],
        [1, 2],
        [1, 2]])
